# Grant-keyed encryption — a hands-on walkthrough

Run this notebook top to bottom, one cell at a time. Each **markdown cell explains one idea**; the
**code cell right after it does exactly that one thing**, using the repo's readable reference library
([`lib/grant_enc.py`](../lib/grant_enc.py) + [`lib/demo.py`](../lib/demo.py)). Edit and re-run any cell to poke at it.

> **Jupyter is polyglot.** This notebook uses Python because the reference crypto is Python. A *deployment*
> walkthrough of the same apparatus could use a shell kernel to drive the `goat` CLI (`goat access grant`, `goat pub/sub`).
> The concepts are identical; only the kernel changes.

The cast of parts, and where each lives on a real mesh, is the **Anatomy** table in the repo README. Here we
run the pieces you'd run yourself (identity, the grant, encrypt/decrypt) and stand in for the mesh services
(portal, `grant-keyd`, the Zenoh router) with the in-process objects in `demo.py`.

> **Reference vs production SDK.** The crypto you *read* here is `lib/grant_enc.py`, the readable reference. The `demo.Producer` / `demo.Consumer` roles below run their seal/open on the **production grant-crypto SDK** when it is present under `./sdk/` (run `bin/fetch-sdk.sh`), and on the reference otherwise — the bytes on the wire are identical either way. The setup cell prints which backend is live.

## 0. Setup

Put the repo's `lib/` on the path and import the two modules. Needs `cryptography` (`pip install cryptography`).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'lib'))
import grant_enc as g      # the crypto: envelope, DEK wrap, signed bundle
import demo as d           # the apparatus: key service, audience, producer, consumer
import sdk_bridge as sdk   # endpoint-crypto seam: the SDK when materialized, else the reference
print('loaded — suite', g.SUITE_V1)
print('endpoint backend:', sdk.backend_name())

## 1. Identities and keys

Every participant holds **its own** keypair — never a shared org key. The `member_id` is the write-anchor/CN
the DEK will be wrapped under. On a real mesh this key is your enrollment certificate's key, issued at onboarding;
here we just generate fresh P-256 keypairs.

We make four: the **key service** (it signs bundles), a **producer**, a **member** consumer, and an **outsider**
(a perfectly valid identity that simply won't be in the audience).

In [ ]:
key_service = g.Identity('site-a/key-service')
producer    = g.Identity('weather-feed/prod')
member      = g.Identity('weather-feed/con')
outsider    = g.Identity('outsider/eve')
for who in (key_service, producer, member, outsider):
    print(f'{who.member_id:24}  holds its own private key')

## 2. The owner's grant, and the derived audience

The data owner records **one** grant — 'org `northwind` may read producer K' — and the key service **derives** the
audience from it plus the org roster. It's never a hand-kept list. Notice two things fall out automatically: the
producer is always its own member (so it can get its own key), and a person with no machine key is *noted* but
not wrapped (they'd read on the human plane, through an app).

In [ ]:
roster = d.Roster()
roster.add_machine('acme-sensors', producer)      # the producer belongs to org acme-sensors
roster.add_machine('northwind', member)         # the grantee org's member
roster.add_human('a-browser-only-person-uuid')

grants = [d.Grant(producer=producer.member_id, grantee_kind='org', grantee_id='northwind'),
          d.Grant(producer=producer.member_id, grantee_kind='principal',
                  grantee_id='a-browser-only-person-uuid')]

audience, notes = d.derive_audience(producer.member_id, grants, roster)
print('DEK will be wrapped to:')
for mid in sorted(audience): print('   ', mid)
for n in notes: print('note:', n)

## 3. The key service wraps + signs + publishes the bundle

This is what `grant-keyd` does on the mesh, off the data path: mint one DEK for this generation, wrap it to each
member's key, **sign** the bundle, and publish it on `keys/<release-set>/<generation>`. We use an in-process bus
that behaves like brokerless Zenoh pub/sub (publishing *retains* the bundle so a late reader can fetch it).

In [ ]:
ks  = d.KeyService(key_service)
bus = d.InProcessBus()
rs  = producer.member_id
gen = ks.rotate(bus, rs, audience)
topic, payload = bus.get_keys(f'keys/{rs}/')[0]
import json
bundle = g.KeyBundle.from_dict(json.loads(payload))
print(f'published {topic}: {len(bundle.members)} wrapped slots, signed by {bundle.signer!r}')

## 4. The producer is a member — it opens its OWN slot to get the DEK

The producer needs no separate key channel. It's in its own audience, so it fetches the bundle and unwraps its
own slot — the same move a consumer makes — to recover the DEK. Then it seals a message (binding suite +
release-set + generation + **topic** as authenticated data) and publishes the ciphertext.

In [ ]:
prod = d.Producer(producer, rs)
data_topic = f'data/{rs}/obs'
plaintext  = b'STATION 7: wind 040 at 12 kt'
sent_gen = prod.publish(bus, data_topic, plaintext)
print(f'producer sealed + published on {data_topic} under generation {sent_gen}')

## 5. A member opens it; an outsider gets the same bytes and cannot

Both subscribe the same topic and receive the **identical bytes**. The member holds a slot and opens the message;
the outsider holds none and is denied — **by the crypto, not the router**. That's the whole read cut.

In [ ]:
con = d.Consumer(member,   rs, ks.verifying_key)
eve = d.Consumer(outsider, rs, ks.verifying_key)
print('member holds', con.refresh_keys(bus), 'key(s);  outsider holds', eve.refresh_keys(bus), 'key(s)')
bus.subscribe_data(data_topic, con.on_message)
bus.subscribe_data(data_topic, eve.on_message)
prod.publish(bus, data_topic, plaintext)    # publish again, now that both are subscribed
for t, gn, pt in con.opened: print('  member  opened gen', gn, '->', pt.decode())
for t, gn in eve.denied:     print('  outsider DENIED on gen', gn, '(no slot)')

## 6. Revocation is rotation

There's no 'delete key' step. To remove the member, mint the **next generation** without it. The member keeps the
generations it already held (bytes can't be clawed back) but gets nothing new.

In [ ]:
audience_after_revoke = {producer.member_id: producer.public}   # member excluded
gen2 = ks.rotate(bus, rs, audience_after_revoke)
held_before = len(con._deks)
held_after  = con.refresh_keys(bus)      # the member finds no gen-2 slot
print(f'member held {held_before} generation(s); after revoke+refresh it holds {held_after} (unchanged — no gen-{gen2} slot)')
prod.publish(bus, data_topic, b'gen-2: only current members can read this')
opened_gens  = sorted({gn for _, gn, _ in con.opened})
denied_gens  = sorted({gn for _, gn in con.denied})
print('member opened generations:', opened_gens, '| denied generations:', denied_gens)

## What you just did

- Held **per-individual keys** — nobody shared a private key.
- Turned **one grant** into a derived audience, then a **signed key bundle** on the fabric.
- Sealed a stream as a **producer-that-is-a-member**, opened it as an authorized member, and watched an
  **outsider be denied by the crypto** while receiving the same bytes.
- **Revoked by rotating** to a new generation.

Where to go next: [`docs/envelope-and-keys.md`](../docs/envelope-and-keys.md) for the crypto in depth,
[`docs/data-owner-controls.md`](../docs/data-owner-controls.md) for the authority model, and
[`docs/honesty-matrix.md`](../docs/honesty-matrix.md) for exactly what's real here vs. represented
(short version: the crypto is real and just ran; the mesh services and PKI are stood in for).

To carry the *same bytes* over a real Zenoh router, see [`local-fabric/README.md`](../local-fabric/README.md).